# KerasTuner Hyperparameter Optimization Example

## 1. Setup and Installation

First, we install `keras-tuner`, a library for hyperparameter optimization developed by Google.

In [1]:
!pip install keras-tuner -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.3 MB/s eta 0:00:00


## 2. Define the Model Building Function

This function constructs a Keras model given a `keras_tuner.HyperParameters` object. It defines a simple Multi-Layer Perceptron (MLP) with one hidden layer. The number of units in the hidden layer (`units`) and the optimizer (`optimizer`) are hyperparameters to be tuned.

In [7]:
from tensorflow import keras
from tensorflow.keras import layers

def build_model(hp):
  units = hp.Int(name="units", min_value=16, max_value=64, step=16)
  model = keras.Sequential([
    layers.Dense(units=units, activation="relu"),
    layers.Dense(10, activation="softmax")
  ])
  optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam"])
  model.compile(
      optimizer=optimizer,
      loss="sparse_categorical_crossentropy",
      metrics=["accuracy"]
  )
  return model

## 3. Define the KerasTuner HyperModel (Alternative)

This section demonstrates an alternative way to define the model for KerasTuner using a `HyperModel` class. This is particularly useful for more complex models or when you want to encapsulate the model building logic within a class.

In [6]:
import keras_tuner as kt

class SimpleMLP(kt.HyperModel):
  def __init__(self, num_classes):
    self.num_classes = num_classes

  def build(self, hp):
    units = hp.Int(name="units", min_value=16, max_value=64, step=16)
    model = keras.Sequential(
        [
         layers.Dense(units=units, activation="relu"),
         layers.Dense(self.num_classes, activation="softmax")
        ]
    )
    optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam"])
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [4]:
hypermodel = SimpleMLP(num_classes=10)

## 4. Defining the Tuner

We initialize `kt.BayesianOptimization` as our tuner. We use the `build_model` function (or `SimpleMLP` if uncommented) to create models, targeting `val_accuracy` as the objective. `max_trials` sets the number of hyperparameter combinations to try, and `executions_per_trial` specifies how many times each combination is run to account for variance.

In [8]:
##Defining the Tuner
tuner = kt.BayesianOptimization(
    build_model,
    objective="val_accuracy",
    max_trials=100,
    executions_per_trial=2,
    directory="mnist_kt_test",
    overwrite=True
)

In [9]:
tuner.search_space_summary()

Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 16, 'max_value': 64, 'step': 16, 'sampling': 'linear'}
optimizer (Choice)
{'default': 'rmsprop', 'conditions': [], 'values': ['rmsprop', 'adam'], 'ordered': False}


## 5. Tuner Search Space Summary

This displays the hyperparameters that KerasTuner will be optimizing, along with their defined ranges or choices.

In [10]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape((-1, 28 * 28)).astype("float32") / 255
X_test = X_test.reshape((-1, 28 * 28)).astype("float32") / 255
X_train_full = X_train[:]
y_train_full = y_train[:]

num_val_samples = 10000
x_train, x_val = X_train[:-num_val_samples], X_train[-num_val_samples:]
y_train, y_val = y_train[:-num_val_samples], y_train[-num_val_samples:]

callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=5)]

tuner.search(
    x_train,
    y_train,
    batch_size=128,
    epochs=100,
    validation_data=(x_val, y_val),
    callbacks=callbacks,
    verbose=2
)

Trial 100 Complete [00h 01m 11s]
val_accuracy: 0.9738999903202057

Best val_accuracy So Far: 0.9771499931812286
Total elapsed time: 02h 05m 35s


## 6. Loading and Preprocessing Data & Running Tuner Search

We load the MNIST dataset, preprocess it by flattening the images and normalizing pixel values. We also split the training data into training and validation sets. Then, we run the hyperparameter search using the `tuner.search` method.

In [11]:
top_n = 4
best_hps = tuner.get_best_hyperparameters(top_n)

## 7. Retrieving Best Hyperparameters

After the search completes, we can retrieve the best performing hyperparameter combinations. `top_n` specifies how many of the best combinations to retrieve.

In [12]:
#getting the best epoch
def get_best_epoch(hp):
  model = build_model(hp)
  callbacks = [
      keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=10)
  ]
  history = model.fit(
      x_train,
      y_train,
      validation_data=(x_val, y_val),
      epochs=100,
      batch_size=128,
      callbacks=callbacks
  )
  val_loss_per_epoch = history.history["val_loss"]
  best_epoch = val_loss_per_epoch.index(min(val_loss_per_epoch)) + 1
  print(f"Best epoch: {best_epoch}")
  return best_epoch

## 8. Function to Get Best Epoch

This helper function trains a model with a given set of hyperparameters and determines the optimal number of training epochs using early stopping. This helps prevent overfitting by stopping training when validation loss stops improving.

In [14]:
def get_best_trained_model(hp):
  model = build_model(hp) # Build the model with the given hyperparameters
  best_epoch = get_best_epoch(hp)
  model.fit(
      X_train_full,
      y_train_full,
      batch_size=128,
      epochs=int(best_epoch * 1.2)
  )
  return model

best_models = []
for hp in best_hps:
  model = get_best_trained_model(hp)
  model.evaluate(X_test, y_test)
  best_models.append(model)

Epoch 1/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8887 - loss: 0.4174 - val_accuracy: 0.9399 - val_loss: 0.2223
Epoch 2/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9388 - loss: 0.2117 - val_accuracy: 0.9528 - val_loss: 0.1730
Epoch 3/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9528 - loss: 0.1627 - val_accuracy: 0.9604 - val_loss: 0.1472
Epoch 4/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9619 - loss: 0.1329 - val_accuracy: 0.9636 - val_loss: 0.1294
Epoch 5/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9669 - loss: 0.1126 - val_accuracy: 0.9671 - val_loss: 0.1161
Epoch 6/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9724 - loss: 0.0979 - val_accuracy: 0.9651 - val_loss: 0.1150
Epoch 7/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9751 - loss: 0.0863 - val_accuracy: 0.9681 - val_loss: 0.1110
Epoch 8/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9776 - loss: 0.0770 - val_accu

## 9. Training Best Models on Full Dataset

Here, we iterate through the `best_hps` (best hyperparameter sets) obtained from the tuner. For each set, we train a new model on the *full* training dataset (`X_train_full`, `y_train_full`) for an optimized number of epochs (determined by `get_best_epoch` with a small buffer). Finally, each trained model is evaluated on the test set and stored.

In [15]:
best_models2 = tuner.get_best_models(top_n)

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


## 10. Getting Best Models from Tuner

Alternatively, KerasTuner can directly provide the best models (not just hyperparameters) that were found during the search process, trained on the best epoch for each trial.

In [16]:
best_models2[0].summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,890 (198.79 KB)

 Trainable params: 50,890 (198.79 KB)

 Non-trainable params: 0 (0.00 B)

## 11. Inspecting the Best Model

We can inspect the architecture of one of the best models identified by the tuner.

## Conclusive Summary

This notebook demonstrates how to effectively use KerasTuner for hyperparameter optimization of a simple neural network on the MNIST dataset. We covered:

*   **Defining a model:** Using a function that takes a `HyperParameters` object.
*   **Setting up the Tuner:** Utilizing `BayesianOptimization` to efficiently explore the hyperparameter space.
*   **Data Preparation:** Loading and preprocessing the MNIST dataset for training.
*   **Hyperparameter Search:** Executing the search to find optimal `units` and `optimizer`.
*   **Post-Search Analysis:** Retrieving the best hyperparameters, identifying the best training epoch using early stopping, and retraining models on the full dataset.

This systematic approach ensures that the final models are robust and perform well by optimizing their configuration beyond manual tuning.